In [12]:
import torch
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from torch import nn
import glob
import seaborn as sns
from PIL import Image
import torchvision.transforms as transforms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import dendropy

import sys
sys.path.append('..')  # Add parent directory to path
from src.birdsbirdsbirds_v2 import GITradeoffModel, evaluate_generalization, evaluate_identification, calculate_average_ball_measure, calculate_alpha_term, theoretical_G_score, theoretical_I_score


In [3]:
# Function to load a model
def load_model(model_path, num_classes, alpha):
    model = GITradeoffModel(num_classes=num_classes, alpha=alpha)
    checkpoint = torch.load(model_path, map_location=torch.device('cpu'))
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()  # Set to evaluation mode
    return model, checkpoint

# Load models for different alpha values
base_dir = "/home/jefe/repos/generalization_transformer/data/models/"
num_classes = 61  # species trained on

# Dictionary to store loaded models by alpha value
models = {}

# Find and load all model files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.startswith("model_alpha") and file.endswith(".pt"):
            # Extract alpha value from filename
            alpha_str = file.split("model_alpha")[1].split("_")[0]
            alpha = float(alpha_str)
            
            # Extract epoch number
            epoch = int(file.split("epoch")[1].split(".")[0])
            
            full_path = os.path.join(root, file)
            #print(f"Loading model with alpha={alpha}, epoch={epoch}")
            
            # Load the model
            model, checkpoint = load_model(full_path, num_classes, alpha)
            
            # Store in dictionary
            if alpha not in models:
                models[alpha] = {}
            models[alpha][epoch] = {
                'model': model,
                'checkpoint': checkpoint,
                'path': full_path
            }

print(f"Loaded {len(models)} different alpha values")

/home/jefe/miniconda3/envs/miller_law/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jefe/miniconda3/envs/miller_law/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialized with 23.63M parameters
Model initialize

In [4]:
# Load models for all alpha values and extract G-I scores
alphas = [0.0, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
results = {}

for alpha in alphas:
    # Find path for this alpha value (update timestamp pattern)
    model_dir = f"/home/jefe/repos/generalization_transformer/data/models/2025*"
    alpha_models = [f for f in glob.glob(f"{model_dir}/model_alpha{alpha}_*.pt")]
    
    # Track scores across epochs
    g_scores = []
    i_scores = []
    ood_g_scores = []
    
    for model_path in sorted(alpha_models):
        checkpoint = torch.load(model_path, map_location='cpu')
        g_scores.append(checkpoint['g_score'])
        i_scores.append(checkpoint['i_score'])
        ood_g_scores.append(checkpoint.get('ood_g_score', 0))  # May not exist in all checkpoints
    
    results[alpha] = {
        'g_scores': g_scores,
        'i_scores': i_scores,
        'ood_g_scores': ood_g_scores
    }

# Plot G-I tradeoff curve (final epoch values)
plt.figure(figsize=(10, 8))
g_values = [results[a]['g_scores'][-1] for a in alphas]
i_values = [results[a]['i_scores'][-1] for a in alphas]
plt.scatter(g_values, i_values, c=alphas, cmap='viridis', s=100)

# Add alpha labels
for i, a in enumerate(alphas):
    plt.annotate(f"α={a}", (g_values[i], i_values[i]), xytext=(5, 5), 
                textcoords='offset points', fontsize=12)

plt.colorbar(label='Alpha Value')
plt.xlabel('Generalization (G)', fontsize=14)
plt.ylabel('Identification (I)', fontsize=14)
plt.title('G-I Tradeoff by Alpha Value', fontsize=16)
plt.grid(False)
plt.savefig('training_trajectories.png')

In [5]:
# Define alpha and threshold values we want to plot
alpha_values = [0.0, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
threshold_values = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32]

# Create figure
plt.figure(figsize=(12, 10))

# Color map for thresholds
colors = plt.cm.viridis(np.linspace(0, 1, len(threshold_values)))

# Initialize data storage 
threshold_data = {t: {'g_scores': [], 'i_scores': [], 'alphas': []} for t in threshold_values}

# Find and load CSV files for each alpha
for alpha in alpha_values:
    # Find the CSV file for this alpha
    csv_files = glob.glob(f"/home/jefe/repos/generalization_transformer/results/models_*/threshold_results_alpha{alpha}.csv")
    
    if not csv_files:
        print(f"No CSV file found for alpha={alpha}")
        continue
    
    # Use the first match (should only be one per alpha)
    csv_file = csv_files[0]
    print(f"Loading data for alpha={alpha} from {os.path.basename(csv_file)}")
    
    # Read the CSV file
    df = pd.read_csv(csv_file)
    
    # Extract data for each threshold
    for threshold in threshold_values:
        # Find the closest threshold in the CSV
        closest_threshold = df['threshold'].iloc[(df['threshold'] - threshold).abs().argsort()[0]]
        
        if abs(closest_threshold - threshold) > 1.0:
            print(f"Warning: Using threshold={closest_threshold} instead of {threshold} for alpha={alpha}")
        
        # Get data for this threshold
        row = df[df['threshold'] == closest_threshold].iloc[0]
        
        # Add to our dataset
        threshold_data[threshold]['g_scores'].append(row['g_score'])
        threshold_data[threshold]['i_scores'].append(row['i_score'])
        threshold_data[threshold]['alphas'].append(alpha)

# Plot a line for each threshold
for i, threshold in enumerate(threshold_values):
    data = threshold_data[threshold]
    
    # Check if we have data for this threshold
    if not data['g_scores']:
        print(f"No data for threshold={threshold}")
        continue
    
    # Plot the G-I tradeoff line for this threshold
    plt.plot(data['g_scores'], data['i_scores'], 'o-', color=colors[i], linewidth=2, 
            label=f'ε={threshold}')
    
    # Annotate the first and last points with alpha values
    plt.annotate(f"α={data['alphas'][0]}", (data['g_scores'][0], data['i_scores'][0]), 
                xytext=(-20, -10), textcoords='offset points', fontsize=9)
    plt.annotate(f"α={data['alphas'][-1]}", (data['g_scores'][-1], data['i_scores'][-1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Formatting
plt.xlabel('Generalization (G)', fontsize=14)
plt.ylabel('Identification (I)', fontsize=14)
plt.title('G-I Tradeoff for Different Thresholds', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(title='Threshold (ε)', bbox_to_anchor=(1.05, 1), loc='upper left')

# Set axis limits
plt.xlim(0.45, 0.7)
plt.ylim(0.45, 0.95)

plt.tight_layout()
plt.savefig('gi_tradeoff_by_threshold.png', dpi=300)

Loading data for alpha=0.0 from threshold_results_alpha0.0.csv
Loading data for alpha=0.25 from threshold_results_alpha0.25.csv
Loading data for alpha=0.5 from threshold_results_alpha0.5.csv
Loading data for alpha=0.75 from threshold_results_alpha0.75.csv
Loading data for alpha=0.8 from threshold_results_alpha0.8.csv
Loading data for alpha=0.85 from threshold_results_alpha0.85.csv
Loading data for alpha=0.9 from threshold_results_alpha0.9.csv
Loading data for alpha=0.95 from threshold_results_alpha0.95.csv
Loading data for alpha=1.0 from threshold_results_alpha1.0.csv


In [6]:
# Create figures for percentage improvement plots
fig1, axes1 = plt.subplots(1, 2, figsize=(16, 8))
fig2 = plt.figure(figsize=(12, 10))

# Plot G and I improvements separately
for i, threshold in enumerate(threshold_values):
    data = threshold_data[threshold]
    
    # Skip if we don't have enough data or alpha=0 is missing
    if not data['g_scores'] or 0.0 not in data['alphas']:
        continue
    
    # Get baseline scores at alpha=0
    baseline_index = data['alphas'].index(0.0)
    baseline_g = data['g_scores'][baseline_index]
    baseline_i = data['i_scores'][baseline_index]
    
    # Calculate percentage improvements
    non_baseline_indices = [j for j, a in enumerate(data['alphas']) if a != 0.0]
    alphas = [data['alphas'][j] for j in non_baseline_indices]
    g_improvements = [(data['g_scores'][j] - baseline_g) / baseline_g * 100 for j in non_baseline_indices]
    i_improvements = [(data['i_scores'][j] - baseline_i) / baseline_i * 100 for j in non_baseline_indices]
    
    # Plot G improvements (left subplot)
    axes1[0].plot(alphas, g_improvements, 'o-', color=colors[i], linewidth=2, label=f'ε={threshold}')
    
    # Plot I improvements (right subplot)
    axes1[1].plot(alphas, i_improvements, 'o-', color=colors[i], linewidth=2, label=f'ε={threshold}')
    
    # Plot G vs I improvements (combined figure)
    plt.figure(fig2.number)
    plt.plot(g_improvements, i_improvements, 'o-', color=colors[i], linewidth=2, label=f'ε={threshold}')
    
    # Annotate some points with alpha values
    for j, alpha in enumerate(alphas):
        if alpha in [0.5, 0.75, 1.0]:  # Select specific alpha values to annotate
            plt.figure(fig2.number)
            plt.annotate(f"α={alpha}", 
                        (g_improvements[j], i_improvements[j]),
                        xytext=(5, 0), textcoords='offset points', fontsize=9)

# Format individual improvement plots
for ax, title, ylabel in zip(axes1, 
                            ['Generalization Improvement', 'Identification Improvement'],
                            ['G Score Improvement (%)', 'I Score Improvement (%)']):
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Alpha (α)', fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.set_title(f'{title} Relative to α=0', fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.legend(title='Threshold (ε)', loc='best')

# Format combined improvement plot
plt.figure(fig2.number)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='black', linestyle='--', alpha=0.3)
plt.xlabel('Generalization (G) Score Improvement (%)', fontsize=14)
plt.ylabel('Identification (I) Score Improvement (%)', fontsize=14)
plt.title('G-I Improvement Tradeoff Relative to α=0', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(title='Threshold (ε)', bbox_to_anchor=(1.05, 1), loc='upper left')

# Save plots
fig1.tight_layout()
fig1.savefig('gi_separate_improvements.png', dpi=300)

plt.figure(fig2.number)
plt.tight_layout()
plt.savefig('gi_combined_improvement.png', dpi=300)


In [7]:
# Create a 3D figure
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(14, 12))
ax = fig.add_subplot(111, projection='3d')

# Plot a 3D line for each threshold
for i, threshold in enumerate(threshold_values):
    data = threshold_data[threshold]
    
    # Check if we have data for this threshold
    if not data['g_scores']:
        continue
    
    # Plot with G on x-axis, Alpha on y-axis, I on z-axis
    # This creates a clearer visualization of the tradeoff
    ax.plot(data['g_scores'], data['alphas'], data['i_scores'], '-', 
            color=colors[i], linewidth=5, alpha=0.5, label=f'ε={threshold}')

# Formatting
ax.set_xlabel('Generalization (G)', fontsize=14)
ax.set_ylabel('Alpha (α)', fontsize=14)
ax.set_zlabel('Identification (I)', fontsize=14)
ax.set_title('G-I-Alpha Tradeoff for Different Thresholds', fontsize=16)

# Set axis limits
ax.set_xlim(0.45, 0.7)
ax.set_ylim(0, 1.0)
ax.set_zlim(0.45, 0.95)

# Set a better viewing angle to highlight the tradeoff
ax.view_init(elev=25, azim=-60)

# Add a legend
ax.legend(title='Threshold (ε)', loc='upper left', bbox_to_anchor=(1.1, 1))

plt.tight_layout()
plt.savefig('gi_alpha_tradeoff_3d.png', dpi=300)


In [8]:
# Function to load models
def load_models(alpha_values=None, epoch=15):
    """Load models for different alpha values"""
    if alpha_values is None:
        alpha_values = [0.0, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
    
    models_dict = {}
    
    for alpha in alpha_values:
        # Find model path
        model_paths = glob.glob(f"/home/jefe/repos/generalization_transformer/data/models/*/model_alpha{alpha}_epoch{epoch}.pt")
        
        if not model_paths:
            print(f"No model found for alpha={alpha}, epoch={epoch}")
            continue
        
        model_path = model_paths[0]
        print(f"Loading model from: {model_path}")
        
        # First load the checkpoint to determine the correct number of classes
        checkpoint = torch.load(model_path, map_location='cpu')
        
        # Determine number of classes from the shape of classifier weights
        num_classes = checkpoint['model_state_dict']['classifier.weight'].shape[0]
        print(f"  Detected {num_classes} classes in the model")
        
        # Create model with correct alpha and number of classes
        model = GITradeoffModel(num_classes=num_classes, alpha=alpha)
        
        # Load weights
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        models_dict[alpha] = model
    
    print(f"Loaded {len(models_dict)} models")
    return models_dict

# Function to extract features from images
def extract_features(model, image_paths):
    """Extract features from images using a model"""
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    all_features = []
    img_names = []
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('RGB')
            img_tensor = transform(img).unsqueeze(0)
            
            with torch.no_grad():
                output = model(img_tensor)
                features = output['features'].squeeze().cpu().numpy()
                
            all_features.append(features)
            img_names.append(os.path.basename(img_path))
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
    
    return np.array(all_features), img_names

# Function to get sample images
def get_sample_images(num_species=10, images_per_species=3):
    """Get sample images from the CUB dataset"""
    sample_images = []
    cub_dir = '../CUB_200_2011/images'

    if os.path.exists(cub_dir):
        print(f"Looking for images in {cub_dir}...")
        
        # Try to get images from different species for diversity
        species_count = 0
        for species_dir in sorted(os.listdir(cub_dir))[:20]:  # First 20 species
            full_species_dir = os.path.join(cub_dir, species_dir)
            if os.path.isdir(full_species_dir):
                species_count += 1
                # Get a few images from this species
                for img_file in sorted(os.listdir(full_species_dir))[:images_per_species]:
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(full_species_dir, img_file)
                        sample_images.append(img_path)
                
                # Limit to requested number of species
                if species_count >= num_species:
                    break

    print(f"Found {len(sample_images)} sample images")
    for i, path in enumerate(sample_images[:5]):  # Show first 5
        print(f"  {i+1}. {path}")
    
    return sample_images

# Function to visualize features using dimensionality reduction
def visualize_features(features_dict, method='pca', perplexity=30):
    """Visualize features using PCA or t-SNE"""
    plt.figure(figsize=(12, 10))
    
    # Combine all features for fitting
    all_features = []
    for alpha, feats in features_dict.items():
        all_features.extend(feats)
    
    all_features = np.vstack(all_features)
    
    # Apply dimensionality reduction
    if method.lower() == 'pca':
        reducer = PCA(n_components=2)
        reduced_features = reducer.fit_transform(all_features)
        title = 'PCA Feature Visualization'
    elif method.lower() == 'tsne':
        reducer = TSNE(n_components=2, perplexity=perplexity, random_state=42)
        reduced_features = reducer.fit_transform(all_features)
        title = f't-SNE Feature Visualization (perplexity={perplexity})'
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Split back by alpha and plot
    alphas = sorted(features_dict.keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(alphas)))
    
    idx = 0
    for i, alpha in enumerate(alphas):
        n_feats = len(features_dict[alpha])
        alpha_feats = reduced_features[idx:idx+n_feats]
        idx += n_feats
        
        plt.scatter(alpha_feats[:, 0], alpha_feats[:, 1], 
                   color=colors[i], label=f'α={alpha}', alpha=0.7, s=80)
    
    plt.title(title, fontsize=16)
    plt.legend(fontsize=12)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'feature_visualization_{method}.png', dpi=300)
    # plt.show()

# Function to compare feature distributions
def compare_feature_distributions(features_dict):
    """Compare statistical properties of features across models"""
    alphas = sorted(features_dict.keys())
    
    # Calculate feature norms
    norms = {alpha: np.linalg.norm(feats, axis=1) for alpha, feats in features_dict.items()}
    
    # Plot norm distributions
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    for i, alpha in enumerate(alphas):
        sns.kdeplot(norms[alpha], label=f'α={alpha}')
    plt.title('Feature Vector Norm Distributions')
    plt.xlabel('Norm')
    plt.legend()
    
    # Plot mean feature values
    plt.subplot(1, 2, 2)
    means = [np.mean(features_dict[alpha], axis=0) for alpha in alphas]
    
    # Sample 50 random dimensions for visualization
    if means[0].shape[0] > 50:
        indices = np.random.choice(means[0].shape[0], 50, replace=False)
        means = [m[indices] for m in means]
    
    # Create heatmap
    plt.imshow(means, aspect='auto', cmap='viridis')
    plt.colorbar(label='Mean Feature Value')
    plt.yticks(range(len(alphas)), [f'α={alpha}' for alpha in alphas])
    plt.title('Mean Feature Values (sample dimensions)')
    
    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=300)
    # plt.show()

# Function to analyze similarity between models
def analyze_feature_similarity(features_dict):
    """Compare similarities between feature representations of different models"""
    alphas = sorted(features_dict.keys())
    
    # Calculate cosine similarities
    similarity_matrix = np.zeros((len(alphas), len(alphas)))
    
    for i, alpha1 in enumerate(alphas):
        for j, alpha2 in enumerate(alphas):
            # Calculate cosine similarity for each image
            sims = []
            for k in range(len(features_dict[alpha1])):
                feat1 = features_dict[alpha1][k]
                feat2 = features_dict[alpha2][k]
                sim = np.dot(feat1, feat2) / (np.linalg.norm(feat1) * np.linalg.norm(feat2))
                sims.append(sim)
            similarity_matrix[i, j] = np.mean(sims)
    
    # Plot similarity matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(similarity_matrix, annot=True, fmt=".2f", 
                xticklabels=[f'α={alpha}' for alpha in alphas],
                yticklabels=[f'α={alpha}' for alpha in alphas],
                cmap='viridis')
    plt.title('Feature Representation Similarity Between Models')
    plt.tight_layout()
    plt.savefig('feature_similarity.png', dpi=300)
    # plt.show()

In [10]:
def measure_phylogenetic_alignment(models_dict, evo_distances, species_indices, common_to_scientific):
    """
    Measure alignment between model representation similarity and phylogenetic similarity
    
    Args:
        models_dict: Dictionary of models with different alpha values
        evo_distances: Matrix of evolutionary distances from phylogenetic tree
        species_indices: Mapping of species names to indices in evo_distances
        common_to_scientific: Mapping from common names to scientific names
    
    Returns:
        Dictionary of alignment scores for each alpha value
    """
    alphas = sorted(models_dict.keys())
    alignment_scores = {
        'spearman': [],
        'kendall': [],
        'pearson': [],
        'alpha_values': alphas
    }
    
    # Get sample images from several species
    cub_dir = '../CUB_200_2011/images'
    species_data = {}
    
    # Get a representative set of species that have evolutionary data
    for species_dir in sorted(os.listdir(cub_dir)):
        species_name = species_dir.split('.')[1]
        if species_name in common_to_scientific:
            sci_name = common_to_scientific[species_name]
            if sci_name in species_indices:
                full_species_dir = os.path.join(cub_dir, species_dir)
                if os.path.isdir(full_species_dir):
                    # Get a few images from this species
                    img_paths = []
                    for img_file in sorted(os.listdir(full_species_dir))[:5]:  # 5 images per species
                        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                            img_path = os.path.join(full_species_dir, img_file)
                            img_paths.append(img_path)
                    
                    if img_paths:
                        species_data[species_name] = {
                            'img_paths': img_paths,
                            'scientific_name': sci_name,
                            'index': species_indices[sci_name]
                        }
                        if len(species_data) >= 20:  # Limit to 20 species for efficiency
                            break
    
    print(f"Found {len(species_data)} species with evolutionary data")
    
    # Process each alpha value
    for alpha in alphas:
        model = models_dict[alpha]
        model_features = {}
        
        # Extract features for each species
        for species_name, data in species_data.items():
            features, _ = extract_features(model, data['img_paths'])
            model_features[species_name] = np.mean(features, axis=0)  # Average features across images
        
        # Calculate pairwise similarities in feature space
        species_list = list(model_features.keys())
        n_species = len(species_list)
        feature_sim_matrix = np.zeros((n_species, n_species))
        evo_dist_matrix = np.zeros((n_species, n_species))
        
        # Fill both matrices
        for i in range(n_species):
            for j in range(n_species):
                sp1, sp2 = species_list[i], species_list[j]
                
                # Feature similarity (cosine similarity)
                feat1, feat2 = model_features[sp1], model_features[sp2]
                sim = np.dot(feat1, feat2) / (np.linalg.norm(feat1) * np.linalg.norm(feat2))
                feature_sim_matrix[i, j] = sim
                
                # Evolutionary distance
                idx1 = species_data[sp1]['index']
                idx2 = species_data[sp2]['index']
                evo_dist_matrix[i, j] = evo_distances[idx1, idx2]
        
        # Convert evolutionary distances to similarities (higher is more similar)
        # Normalize to [0,1] range and invert
        evo_dist_max = np.max(evo_dist_matrix)
        if evo_dist_max > 0:
            evo_sim_matrix = 1 - (evo_dist_matrix / evo_dist_max)
        else:
            evo_sim_matrix = np.zeros_like(evo_dist_matrix)
        
        # Flatten the matrices for correlation (use upper triangular part only)
        feature_sims = feature_sim_matrix[np.triu_indices(n_species, k=1)]
        evo_sims = evo_sim_matrix[np.triu_indices(n_species, k=1)]
        
        # Calculate correlations
        spearman = scipy.stats.spearmanr(feature_sims, evo_sims).correlation
        kendall = scipy.stats.kendalltau(feature_sims, evo_sims).correlation
        pearson = scipy.stats.pearsonr(feature_sims, evo_sims).statistic
        
        alignment_scores['spearman'].append(spearman)
        alignment_scores['kendall'].append(kendall)
        alignment_scores['pearson'].append(pearson)
        
        print(f"Alpha {alpha}: Spearman={spearman:.3f}, Kendall={kendall:.3f}, Pearson={pearson:.3f}")
    
    # Plot alignment scores
    plt.figure(figsize=(12, 8))
    plt.plot(alphas, alignment_scores['spearman'], 'o-', linewidth=2, label='Spearman Correlation')
    plt.plot(alphas, alignment_scores['kendall'], 's-', linewidth=2, label='Kendall Tau')
    plt.plot(alphas, alignment_scores['pearson'], '^-', linewidth=2, label='Pearson Correlation')
    
    plt.xlabel('Alpha (α)', fontsize=14)
    plt.ylabel('Correlation with Phylogenetic Similarity', fontsize=14)
    plt.title('Representation-Phylogenetic Alignment by Alpha Value', fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig('phylogenetic_alignment.png', dpi=300)
    
    # Create scatter plots for selected alpha values
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    selected_alphas = [0.0, 0.5, 1.0] if len(alphas) >= 3 else alphas[:min(3, len(alphas))]
    
    for i, alpha in enumerate(selected_alphas):
        idx = alphas.index(alpha)
        model = models_dict[alpha]
        
        # Get feature similarities and evolutionary similarities for this alpha
        feature_sims = feature_sim_matrix[np.triu_indices(n_species, k=1)]
        evo_sims = evo_sim_matrix[np.triu_indices(n_species, k=1)]
        
        # Plot scatter
        axes[i].scatter(evo_sims, feature_sims, alpha=0.6)
        axes[i].set_xlabel('Phylogenetic Similarity', fontsize=12)
        axes[i].set_ylabel('Feature Similarity', fontsize=12)
        axes[i].set_title(f'α={alpha} (corr={alignment_scores["spearman"][idx]:.3f})', fontsize=14)
        axes[i].grid(True, alpha=0.3)
        
        # Add best fit line
        z = np.polyfit(evo_sims, feature_sims, 1)
        p = np.poly1d(z)
        axes[i].plot(np.sort(evo_sims), p(np.sort(evo_sims)), "r--", alpha=0.7)
    
    plt.tight_layout()
    plt.savefig('phylogenetic_scatter_plots.png', dpi=300)
    
    return alignment_scores

# Usage example:
# Load models
models_dict = load_models()

# Import necessary libraries
import scipy.stats

# Run the analysis
alignment_scores = measure_phylogenetic_alignment(
    models_dict, 
    evo_distances,  # This comes from your original script
    species_indices,  # This mapping comes from your original script
    common_to_scientific  # This mapping comes from your original script
)

# You can also visualize the percentage improvement relative to alpha=0
if 0.0 in alignment_scores['alpha_values']:
    baseline_idx = alignment_scores['alpha_values'].index(0.0)
    baseline_spearman = alignment_scores['spearman'][baseline_idx]
    baseline_kendall = alignment_scores['kendall'][baseline_idx]
    baseline_pearson = alignment_scores['pearson'][baseline_idx]
    
    # Calculate improvements
    spearman_improve = [(score - baseline_spearman) / abs(baseline_spearman) * 100 
                        for score in alignment_scores['spearman']]
    kendall_improve = [(score - baseline_kendall) / abs(baseline_kendall) * 100 
                       for score in alignment_scores['kendall']]
    pearson_improve = [(score - baseline_pearson) / abs(baseline_pearson) * 100 
                       for score in alignment_scores['pearson']]
    
    # Plot improvements
    plt.figure(figsize=(12, 8))
    plt.plot(alignment_scores['alpha_values'], spearman_improve, 'o-', linewidth=2, label='Spearman')
    plt.plot(alignment_scores['alpha_values'], kendall_improve, 's-', linewidth=2, label='Kendall')
    plt.plot(alignment_scores['alpha_values'], pearson_improve, '^-', linewidth=2, label='Pearson')
    
    plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    plt.xlabel('Alpha (α)', fontsize=14)
    plt.ylabel('Correlation Improvement (%)', fontsize=14)
    plt.title('Phylogenetic Alignment Improvement Relative to α=0', fontsize=16)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig('phylogenetic_alignment_improvement.png', dpi=300)

Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250509_173818/model_alpha0.0_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_020747/model_alpha0.25_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_033148/model_alpha0.5_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_045704/model_alpha0.75_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_062208/model_alpha0.8_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/je

NameError: name 'evo_distances' is not defined

In [16]:
def measure_phylogenetic_alignment(models_dict, evo_distances, species_indices, common_to_scientific):
    """Measure alignment between model representation similarity and phylogenetic similarity"""
    alphas = sorted(models_dict.keys())
    alignment_scores = {'spearman': [], 'kendall': [], 'pearson': [], 'alpha_values': alphas}
    
    # Debug info
    print(f"Evolutionary distance matrix shape: {evo_distances.shape}")
    max_idx = evo_distances.shape[0] - 1
    print(f"Maximum valid index: {max_idx}")
    
    # IMPORTANT: Filter species indices to valid range
    valid_indices = {}
    for sci_name, idx in species_indices.items():
        if idx <= max_idx:
            valid_indices[sci_name] = idx
        else:
            print(f"Filtering out {sci_name} with invalid index {idx}")
    
    print(f"Filtered indices from {len(species_indices)} to {len(valid_indices)}")
    
    # Build a filtered mapping of common to scientific names
    valid_common_to_sci = {}
    for common, sci in common_to_scientific.items():
        if sci in valid_indices:
            valid_common_to_sci[common] = sci
    
    print(f"Valid common name mappings: {len(valid_common_to_sci)}")
    
    # Find valid species with images
    cub_dir = '../CUB_200_2011/images'
    species_data = {}
    
    # Get species that have valid evolutionary data
    for species_dir in sorted(os.listdir(cub_dir)):
        if '.' not in species_dir:
            continue
        
        species_name = species_dir.split('.')[1]
        if species_name in valid_common_to_sci:
            sci_name = valid_common_to_sci[species_name]
            idx = valid_indices[sci_name]
            
            # Double check index is valid
            if idx > max_idx:
                print(f"ERROR: Index {idx} still invalid for {species_name}")
                continue
                
            full_species_dir = os.path.join(cub_dir, species_dir)
            if os.path.isdir(full_species_dir):
                # Get images
                img_paths = []
                for img_file in sorted(os.listdir(full_species_dir))[:5]:
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_paths.append(os.path.join(full_species_dir, img_file))
                
                if img_paths:
                    species_data[species_name] = {
                        'img_paths': img_paths,
                        'scientific_name': sci_name,
                        'index': idx
                    }
    
    print(f"Found {len(species_data)} species with valid data and images")
    
    # Need at least 2 species for correlation
    if len(species_data) < 2:
        print("Not enough valid species to calculate correlations")
        return {key: [0] * len(alphas) for key in alignment_scores}
        
    # Continue with analysis for each alpha
    for alpha in alphas:
        print(f"\nProcessing alpha={alpha}")
        model = models_dict[alpha]
        model_features = {}
        
        # Extract features
        for species_name, data in species_data.items():
            features, _ = extract_features(model, data['img_paths'])
            model_features[species_name] = np.mean(features, axis=0)
            
        species_list = list(model_features.keys())
        n_species = len(species_list)
        feature_sim_matrix = np.zeros((n_species, n_species))
        evo_dist_matrix = np.zeros((n_species, n_species))
        
        # Fill matrices - with additional safety checks
        for i in range(n_species):
            for j in range(n_species):
                sp1, sp2 = species_list[i], species_list[j]
                
                # Feature similarity
                feat1 = model_features[sp1]
                feat2 = model_features[sp2]
                feat_sim = np.dot(feat1, feat2) / (np.linalg.norm(feat1) * np.linalg.norm(feat2))
                feature_sim_matrix[i, j] = feat_sim
                
                # Evolution distance - TRIPLE CHECK INDICES
                idx1 = species_data[sp1]['index']
                idx2 = species_data[sp2]['index']
                
                if idx1 >= evo_distances.shape[0] or idx2 >= evo_distances.shape[0]:
                    print(f"CRITICAL ERROR: Index out of bounds: {idx1},{idx2}")
                    evo_dist_matrix[i, j] = 0
                else:
                    evo_dist_matrix[i, j] = evo_distances[idx1, idx2]
        
        # Convert to similarities
        evo_dist_max = np.max(evo_dist_matrix)
        if evo_dist_max > 0:
            evo_sim_matrix = 1 - (evo_dist_matrix / evo_dist_max)
        else:
            evo_sim_matrix = np.zeros_like(evo_dist_matrix)
        
        # Calculate correlations - upper triangle only
        feature_sims = feature_sim_matrix[np.triu_indices(n_species, k=1)]
        evo_sims = evo_sim_matrix[np.triu_indices(n_species, k=1)]
        
        # Safely calculate correlations
        try:
            spearman = scipy.stats.spearmanr(feature_sims, evo_sims).correlation
            if np.isnan(spearman): spearman = 0
        except Exception as e:
            print(f"Spearman error: {e}")
            spearman = 0
            
        try:
            kendall = scipy.stats.kendalltau(feature_sims, evo_sims).correlation
            if np.isnan(kendall): kendall = 0
        except Exception as e:
            print(f"Kendall error: {e}")
            kendall = 0
            
        try:
            pearson = scipy.stats.pearsonr(feature_sims, evo_sims).statistic
            if np.isnan(pearson): pearson = 0
        except Exception as e:
            print(f"Pearson error: {e}")
            pearson = 0
        
        alignment_scores['spearman'].append(spearman)
        alignment_scores['kendall'].append(kendall)
        alignment_scores['pearson'].append(pearson)
        
        print(f"Alpha {alpha}: Spearman={spearman:.3f}, Kendall={kendall:.3f}, Pearson={pearson:.3f}")
    
    # Create plots if we have valid results
    if any(abs(x) > 0.01 for x in alignment_scores['spearman']):
        # Plot alignment scores
        plt.figure(figsize=(12, 8))
        plt.plot(alphas, alignment_scores['spearman'], 'o-', linewidth=2, label='Spearman')
        plt.plot(alphas, alignment_scores['kendall'], 's-', linewidth=2, label='Kendall')
        plt.plot(alphas, alignment_scores['pearson'], '^-', linewidth=2, label='Pearson')
        
        plt.xlabel('Alpha (α)', fontsize=14)
        plt.ylabel('Correlation with Phylogenetic Similarity', fontsize=14)
        plt.title('Representation-Phylogenetic Alignment by Alpha Value', fontsize=16)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig('phylogenetic_alignment.png', dpi=300)
    
    return alignment_scores

In [21]:

# 1. FIRST, LOAD THE EVOLUTIONARY DISTANCE MATRIX
try:
    # Try to load the saved matrix
    evo_distances = np.load("../data/evo_distance_matrix.npy")
    print(f"Successfully loaded evolutionary distance matrix with shape: {evo_distances.shape}")
except Exception as e:
    print(f"Error loading evolutionary distance matrix: {e}")
    # Create a placeholder if loading fails
    evo_distances = np.ones((71, 71))  # Create a dummy matrix
    print("Created placeholder distance matrix")

# 2. LOAD THE TREE AND CREATE SPECIES MAPPINGS
try:
    # Load the tree
    tree = dendropy.Tree.get(path="../data/birds_species.nwk", schema="newick")
    print(f"Successfully loaded tree with {len(tree.taxon_namespace)} taxa")
    
    # Create species indices mapping
    species_indices = {}
    for i, taxon in enumerate(tree.taxon_namespace):
        if taxon.label:
            species_indices[taxon.label] = i
    
    print(f"Created mapping for {len(species_indices)} scientific names")
except Exception as e:
    print(f"Error loading tree: {e}")
    # Create a placeholder mapping
    species_indices = {}
    print("Created empty species indices mapping")

# 3. LOAD CLASS NAMES AND CREATE COMMON-TO-SCIENTIFIC MAPPING
try:
    # Read CUB class names
    cub_root = '../CUB_200_2011'
    with open(os.path.join(cub_root, 'classes.txt'), 'r') as f:
        class_names = [line.split('.')[1].strip() for line in f.readlines()]
    print(f"Loaded {len(class_names)} class names")
    
    # Create common to scientific mapping
    # Manual mappings - use your actual mapping
    common_to_scientific = {
        'Black_footed_Albatross': 'Phoebastria nigripes',
        'Laysan_Albatross': 'Phoebastria immutabilis',
        'Sooty_Albatross': 'Phoebetria fusca',
        'Groove_billed_Ani': 'Crotophaga sulcirostris',
        'Crested_Auklet': 'Aethia cristatella',
        'Least_Auklet': 'Aethia pusilla',
        'Parakeet_Auklet': 'Aethia psittacula',
        'Rhinoceros_Auklet': 'Cerorhinca monocerata',
        'Brewer_Blackbird': 'Euphagus cyanocephalus',
        'Red_winged_Blackbird': 'Agelaius phoeniceus',
        'Rusty_Blackbird': 'Euphagus carolinus',
        'Yellow_headed_Blackbird': 'Xanthocephalus xanthocephalus',
        'Bobolink': 'Dolichonyx oryzivorus',
        'Indigo_Bunting': 'Passerina cyanea',
        'Lazuli_Bunting': 'Passerina amoena',
        'Painted_Bunting': 'Passerina ciris',
        'Cardinal': 'Cardinalis cardinalis',
    }
    
    print(f"Created mapping with {len(common_to_scientific)} entries")
except Exception as e:
    print(f"Error loading class names: {e}")
    common_to_scientific = {}
    print("Created empty common-to-scientific mapping")

# 4. CHECK VARIABLE STATES BEFORE CALLING FUNCTION
print("\nStatus summary before running analysis:")
print(f"- evo_distances shape: {evo_distances.shape}")
print(f"- species_indices count: {len(species_indices)}")
print(f"- common_to_scientific count: {len(common_to_scientific)}")
print(f"- models_dict loaded: {'Yes' if 'models_dict' in globals() else 'No'}")

# 5. NOW CALL THE FUNCTION WITH ALL VARIABLES EXPLICITLY DEFINED
if 'models_dict' in globals() and len(species_indices) > 0 and len(common_to_scientific) > 0:
    print("\nRunning phylogenetic alignment analysis...")
    
    # Now run the analysis with the loaded data
    alignment_scores = measure_phylogenetic_alignment(
        models_dict, 
        evo_distances,
        species_indices,
        common_to_scientific
    )
    
    # Plot improvement relative to alpha=0
    # You can also visualize the percentage improvement relative to alpha=0
    if 0.0 in alignment_scores['alpha_values']:
        baseline_idx = alignment_scores['alpha_values'].index(0.0)
        baseline_spearman = alignment_scores['spearman'][baseline_idx]
        baseline_kendall = alignment_scores['kendall'][baseline_idx]
        baseline_pearson = alignment_scores['pearson'][baseline_idx]
        
        # Calculate improvements with handling for zero baselines
        spearman_improve = []
        kendall_improve = []
        pearson_improve = []
        
        for score in alignment_scores['spearman']:
            if abs(baseline_spearman) < 0.001:  # Essentially zero
                if score > 0:
                    spearman_improve.append(100)  # Show as 100% improvement
                elif score < 0:
                    spearman_improve.append(-100)  # Show as -100% improvement
                else:
                    spearman_improve.append(0)  # No change
            else:
                spearman_improve.append((score - baseline_spearman) / abs(baseline_spearman) * 100)
        
        for score in alignment_scores['kendall']:
            if abs(baseline_kendall) < 0.001:  # Essentially zero
                if score > 0:
                    kendall_improve.append(100)
                elif score < 0:
                    kendall_improve.append(-100)
                else:
                    kendall_improve.append(0)
            else:
                kendall_improve.append((score - baseline_kendall) / abs(baseline_kendall) * 100)
        
        for score in alignment_scores['pearson']:
            if abs(baseline_pearson) < 0.001:  # Essentially zero
                if score > 0:
                    pearson_improve.append(100)
                elif score < 0:
                    pearson_improve.append(-100)
                else:
                    pearson_improve.append(0)
            else:
                pearson_improve.append((score - baseline_pearson) / abs(baseline_pearson) * 100)
        
        # Plot improvements
        plt.figure(figsize=(12, 8))
        plt.plot(alignment_scores['alpha_values'], spearman_improve, 'o-', linewidth=2, label='Spearman')
        plt.plot(alignment_scores['alpha_values'], kendall_improve, 's-', linewidth=2, label='Kendall')
        plt.plot(alignment_scores['alpha_values'], pearson_improve, '^-', linewidth=2, label='Pearson')
        
        plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        plt.xlabel('Alpha (α)', fontsize=14)
        plt.ylabel('Correlation Improvement (%)', fontsize=14)
        plt.title('Phylogenetic Alignment Improvement Relative to α=0', fontsize=16)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=12)
        plt.tight_layout()
        plt.savefig('phylogenetic_alignment_improvement.png', dpi=300)
        
else:
    print("\nCannot run analysis - missing required variables")
    missing = []
    if 'models_dict' not in globals():
        missing.append("models_dict")
    if len(species_indices) == 0:
        missing.append("species_indices")
    if len(common_to_scientific) == 0:
        missing.append("common_to_scientific")
    print(f"Missing: {', '.join(missing)}")

Successfully loaded evolutionary distance matrix with shape: (71, 71)
Successfully loaded tree with 7149 taxa
Created mapping for 7149 scientific names
Loaded 200 class names
Created mapping with 17 entries

Status summary before running analysis:
- evo_distances shape: (71, 71)
- species_indices count: 7149
- common_to_scientific count: 17
- models_dict loaded: Yes

Running phylogenetic alignment analysis...
Evolutionary distance matrix shape: (71, 71)
Maximum valid index: 70
Filtering out Apus apus with invalid index 71
Filtering out Apus unicolor with invalid index 72
Filtering out Apus alexandri with invalid index 73
Filtering out Apus niansae with invalid index 74
Filtering out Cypsiurus balasiensis with invalid index 75
Filtering out Cypsiurus parvus with invalid index 76
Filtering out Hydrochous gigas with invalid index 77
Filtering out Aerodramus salangana with invalid index 78
Filtering out Aerodramus fuciphagus with invalid index 79
Filtering out Aerodramus brevirostris with 

In [42]:
# 1. Load models
alpha_values = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]  # Subset for clarity
models_dict = load_models(alpha_values)

# 2. Get sample images
sample_images = get_sample_images(num_species=10, images_per_species=3)

# 3. Extract features from all models
features_dict = {}
for alpha, model in models_dict.items():
    print(f"Extracting features for model with α={alpha}...")
    features, _ = extract_features(model, sample_images)
    features_dict[alpha] = features
    print(f"  Extracted features with shape: {features.shape}")

print(f"Extracted features for {len(features_dict)} models")

Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_004410/model_alpha0.0_epoch15.pt
  Detected 61 classes in the model


/home/jefe/miniconda3/envs/miller_law/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jefe/miniconda3/envs/miller_law/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_020747/model_alpha0.25_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_033148/model_alpha0.5_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_045704/model_alpha0.75_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_091115/model_alpha0.9_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63M parameters
Loading model from: /home/jefe/repos/generalization_transformer/data/models/20250504_120102/model_alpha1.0_epoch15.pt
  Detected 61 classes in the model
Model initialized with 23.63

In [43]:
# 4. Visualize features using dimensionality reduction
print("Visualizing features with PCA...")
visualize_features(features_dict, method='pca')

Visualizing features with PCA...


In [44]:
print("Visualizing features with t-SNE...")
visualize_features(features_dict, method='tsne', perplexity=15)

Visualizing features with t-SNE...


In [45]:
# 5. Compare feature distributions
print("Comparing feature distributions...")
compare_feature_distributions(features_dict)

Comparing feature distributions...


In [46]:
# 6. Analyze feature similarity
print("Analyzing feature similarity between models...")
analyze_feature_similarity(features_dict)

Analyzing feature similarity between models...


In [48]:
# Function to compute the similarity function for a model
def compute_similarity_function(model, feature_dict, distances, num_bins=20):
    """
    Compute probability of correct mapping as a function of distance
    
    Args:
        model: The trained model
        feature_dict: Dictionary of features for each class
        distances: Matrix of distances between classes
        num_bins: Number of distance bins to use
    
    Returns:
        distance_bins: Center of each distance bin
        probabilities: Probability of correct mapping for each bin
    """
    # Create distance bins
    min_dist = np.min(distances)
    max_dist = np.max(distances)
    distance_bins = np.linspace(min_dist, max_dist, num_bins + 1)
    bin_centers = (distance_bins[:-1] + distance_bins[1:]) / 2
    
    # Initialize counts for each bin
    correct_counts = np.zeros(num_bins)
    total_counts = np.zeros(num_bins)
    
    # For each pair of classes
    classes = list(feature_dict.keys())
    for i, class1 in enumerate(classes):
        for j, class2 in enumerate(classes):
            if i == j:
                continue
                
            # Get distance between these classes
            dist = distances[i, j]
            
            # Find which bin this distance falls into
            bin_idx = np.digitize(dist, distance_bins) - 1
            if bin_idx >= num_bins:
                continue
                
            # For each feature in class1, compute similarity to features in class2
            for feat1 in feature_dict[class1]:
                for feat2 in feature_dict[class2]:
                    # Use the model to predict probability of correct mapping
                    with torch.no_grad():
                        feat1_tensor = torch.tensor(feat1).unsqueeze(0)
                        feat2_tensor = torch.tensor(feat2).unsqueeze(0)
                        
                        # Compute similarity score
                        similarity = model.compute_similarity(feat1_tensor, feat2_tensor).item()
                        
                        # Count as correct if similarity exceeds threshold
                        if similarity > 0.5:  # Adjust threshold as needed
                            correct_counts[bin_idx] += 1
                        total_counts[bin_idx] += 1
    
    # Compute probabilities for each bin
    probabilities = np.zeros(num_bins)
    for i in range(num_bins):
        if total_counts[i] > 0:
            probabilities[i] = correct_counts[i] / total_counts[i]
    
    return bin_centers, probabilities

# Extract similarity functions for different alpha values
alpha_values = [0.0, 0.25, 0.5, 0.75, 0.9, 1.0]
similarity_functions = {}

# Load evolutionary distance matrix
distance_matrix = evo_distances

# Load or compute features for all species
# This would need to be adjusted based on your actual data structure
class_features = {}  # Dictionary of features for each class

# Compute similarity function for each alpha
for alpha in alpha_values:
    model = models_dict[alpha]
    distances, probabilities = compute_similarity_function(model, class_features, distance_matrix)
    similarity_functions[alpha] = (distances, probabilities)

# Plot the similarity functions
plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(alpha_values)))

for i, alpha in enumerate(alpha_values):
    distances, probabilities = similarity_functions[alpha]
    plt.plot(distances, probabilities, '-o', color=colors[i], 
             linewidth=2, label=f'α={alpha}')

plt.xlabel('Evolutionary Distance', fontsize=14)
plt.ylabel('Probability of Correct Mapping', fontsize=14)
plt.title('Similarity Function Across Different Alpha Values', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('similarity_functions.png', dpi=300)
plt.show()

# 3D visualization of similarity functions
fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection='3d')

# Create a mesh grid for alpha and distance
alpha_mesh, dist_mesh = np.meshgrid(alpha_values, np.linspace(min_dist, max_dist, 50))
prob_mesh = np.zeros_like(alpha_mesh)

# Interpolate probability values for the mesh
for i, alpha in enumerate(alpha_values):
    distances, probabilities = similarity_functions[alpha]
    interp_func = scipy.interpolate.interp1d(distances, probabilities, 
                                            bounds_error=False, fill_value="extrapolate")
    for j in range(prob_mesh.shape[0]):
        prob_mesh[j, i] = interp_func(dist_mesh[j, 0])

# Plot the surface
surf = ax.plot_surface(alpha_mesh, dist_mesh, prob_mesh, cmap='viridis', 
                      edgecolor='none', alpha=0.8)

ax.set_xlabel('Alpha (α)', fontsize=14)
ax.set_ylabel('Evolutionary Distance', fontsize=14)
ax.set_zlabel('Probability of Correct Mapping', fontsize=14)
ax.set_title('Similarity Function Across Different Alpha Values', fontsize=16)
plt.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.tight_layout()
plt.savefig('similarity_function_3d.png', dpi=300)
plt.show()

NameError: name 'evo_distances' is not defined